# MITIC Paraguai

Notebook autônomo da pesquisa FAPESP: executa no Colab sem depender do GitHub.


In [ ]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "requests==2.32.3", "beautifulsoup4==4.12.3", "urllib3==2.2.3", "pandas==2.2.3"], check=True)


In [ ]:
import re, time
from dataclasses import dataclass, asdict
from datetime import datetime, timezone
from urllib.parse import parse_qsl, urlencode, urljoin, urlparse, urlunparse
import requests
from bs4 import BeautifulSoup
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry


@dataclass
class Record:
    source_id: str
    source: str
    title: str
    date: str
    url: str
    summary: str
    collected_at: str


def session() -> requests.Session:
    s = requests.Session()
    retry = Retry(total=3, connect=3, read=3, backoff_factor=1, status_forcelist=(429, 500, 502, 503, 504), allowed_methods=("GET",))
    s.mount("https://", HTTPAdapter(max_retries=retry))
    s.headers.update({"User-Agent": "Mozilla/5.0 (compatible; FAPESP-Research-Scraper/1.0; +https://github.com/anamacao/FAPESP-PIBIC-scrapping)", "Accept-Language": "pt-BR,pt;q=0.9,en;q=0.8,es;q=0.7"})
    return s


def page_url(cfg: dict, page: int) -> str:
    if page == 0:
        return cfg["url"]
    if cfg.get("page_template"):
        return cfg["page_template"].format(page=page + 1)
    if cfg.get("page_param"):
        p = urlparse(cfg["url"])
        query = dict(parse_qsl(p.query))
        query[cfg["page_param"]] = str(page)
        return urlunparse(p._replace(query=urlencode(query)))
    return cfg["url"]


def _text(node) -> str:
    return re.sub(r"\s+", " ", node.get_text(" ", strip=True)).strip()


def _date(container) -> str:
    if not container:
        return ""
    tag = container.select_one("time, [datetime], .date, .data, .field--name-created, .views-field-created, .news-date")
    if not tag:
        return ""
    return (tag.get("datetime") or _text(tag))[:80]


def extract_listing(source_id: str, cfg: dict, html: str, base_url: str) -> list[Record]:
    soup = BeautifulSoup(html, "html.parser")
    candidates = soup.select("article h2 a[href], article h3 a[href], article h4 a[href], .views-row h2 a[href], .views-row h3 a[href], main h2 a[href], main h3 a[href]")
    # O EDPB usa cards em que o título e o link vazio são elementos irmãos.
    card_links = soup.select(".node-article-card a.node-article-card__link[href]")
    for a in card_links:
        card = a.find_parent(class_="node-article-card")
        title_node = card.select_one(".node-article-card__title") if card else None
        if title_node:
            a.string = _text(title_node)
    candidates.extend(card_links)
    # Com filtros de domínio/caminho ativos, examinar todos os links é seguro e
    # cobre portais que não usam elementos semânticos article/main.
    if cfg.get("include"):
        candidates.extend(soup.select("a[href]"))
    elif not candidates:
        candidates = soup.select("main a[href], article a[href]")
    # O portal ICN expõe alguns links apenas como atributos de componentes JS.
    for node in soup.select("[data-contenido-link]"):
        target = node.get("data-contenido-link", "")
        if target and not target.startswith(("http://", "https://")):
            target = f"https://icn.gob.ar/noticias/{target.strip('/')}"
        if target:
            synthetic = soup.new_tag("a", href=target)
            synthetic.string = node.get("data-contenido-link-name") or node.get_text(" ", strip=True) or target.rsplit("/", 1)[-1].replace("-", " ")
            candidates.append(synthetic)
    include = cfg.get("include", [])
    exclude = cfg.get("exclude", [])
    now = datetime.now(timezone.utc).isoformat()
    rows, seen = [], set()
    for a in candidates:
        href = urljoin(base_url, a.get("href", ""))
        title = _text(a)
        if len(title) < 12 or href in seen:
            continue
        if include and not any(token in href for token in include):
            continue
        if any(token in href for token in exclude):
            continue
        container = a.find_parent(["article", "li", "div"])
        summary = ""
        if container:
            p = container.select_one("p, .summary, .field--name-body")
            summary = _text(p)[:1000] if p else ""
        rows.append(Record(source_id, cfg["name"], title, _date(container), href, summary, now))
        seen.add(href)
    # O portal gov.br/ANPD entrega as notícias no estado JSON da aplicação,
    # mesmo quando não renderiza os links no HTML recebido pelo requests.
    for script in soup.find_all("script"):
        raw = script.string or script.get_text() or ""
        if not raw.strip().startswith("window.__data="):
            continue
        try:
            serialized = raw.split("=", 1)[1].strip().rstrip(";")
            serialized = re.sub(r"\bundefined\b", "null", serialized)
            payload = json.loads(serialized)
        except (json.JSONDecodeError, IndexError):
            continue
        stack = [payload]
        while stack:
            item = stack.pop()
            if isinstance(item, dict):
                href = item.get("@id", "")
                title = item.get("title", "")
                if item.get("@type") == "News Item" and href and title and href not in seen:
                    rows.append(Record(source_id, cfg["name"], title, item.get("effective", ""), href, item.get("description", "")[:1000], now))
                    seen.add(href)
                stack.extend(item.values())
            elif isinstance(item, list):
                stack.extend(item)
    return rows


def collect(source_id: str, cfg: dict, delay: float = 0.5) -> tuple[list[dict], dict]:
    s = session()
    records, errors, pages_ok = [], [], 0
    for page in range(cfg.get("pages", 1)):
        url = page_url(cfg, page)
        try:
            response = s.get(url, timeout=30)
            response.raise_for_status()
            pages_ok += 1
            records.extend(extract_listing(source_id, cfg, response.text, response.url))
        except Exception as exc:
            errors.append({"page": page + 1, "url": url, "error": f"{type(exc).__name__}: {exc}"})
        if page + 1 < cfg.get("pages", 1):
            time.sleep(delay)
    unique = {r.url: r for r in records}
    now = datetime.now(timezone.utc).isoformat()
    for title, url in cfg.get("static_links", []):
        unique.setdefault(url, Record(source_id, cfg["name"], title, "", url, "Catálogo oficial; conteúdo carregado dinamicamente.", now))
    status = "ok" if unique else ("failed" if not pages_ok else "empty")
    diagnostic = {"source_id": source_id, "source": cfg["name"], "status": status, "pages_ok": pages_ok, "pages_requested": cfg.get("pages", 1), "records": len(unique), "errors": errors}
    return [asdict(r) for r in unique.values()], diagnostic



source_id = 'mitic_paraguai'
cfg = {'name': 'MITIC Paraguai', 'url': 'https://mitic.gov.py/noticias/', 'pages': 5, 'page_template': 'https://mitic.gov.py/noticias/page/{page}/', 'include': ['mitic.gov.py/']}


In [ ]:
from pathlib import Path
import json
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    display = print

rows, diagnostic = collect(source_id, cfg)
output = Path("/content/fapesp-data") if Path("/content").exists() else Path.cwd() / "fapesp-data"
output.mkdir(parents=True, exist_ok=True)
csv_path = output / f"{source_id}.csv"
pd.DataFrame(rows, columns=["source_id", "source", "title", "date", "url", "summary", "collected_at"]).to_csv(csv_path, index=False)
(output / f"{source_id}-diagnostic.json").write_text(json.dumps(diagnostic, ensure_ascii=False, indent=2), encoding="utf-8")
display(pd.DataFrame(rows).head(20))
display(diagnostic)
print("Arquivos salvos em:", output)
